# 03 — Bronze → Silver

Aplica **Data Quality** (limpeza, padronizacao e tipagem) e grava na camada **Silver**.

In [ ]:
from pyspark.sql.functions import (
    col,
    trim,
    upper,
    lower,
    initcap,
    current_timestamp,
    when
)

BRONZE_SCHEMA = "workspace.bronze"
SILVER_SCHEMA = "workspace.silver"

TABLES = [
    "clientes",
    "produtos",
    "pedidos"
]

In [ ]:
spark.sql(f"""
CREATE SCHEMA IF NOT EXISTS {SILVER_SCHEMA}
""")

results_silver = []

In [ ]:
# CLIENTES
clientes = (
    spark.table(f"{BRONZE_SCHEMA}.clientes")
        .select(
            col("id").cast("string").alias("id"),
            trim(initcap(col("nome"))).alias("nome"),
            trim(lower(col("email"))).alias("email"),
            trim(initcap(col("cidade"))).alias("cidade")
        )
        .dropDuplicates(["id"])
        .filter("id IS NOT NULL")
        .filter("nome IS NOT NULL")
        .filter("email IS NOT NULL")
        .withColumn("_silver_processed_at", current_timestamp())
)

(
    clientes.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{SILVER_SCHEMA}.clientes")
)

results_silver.append({"table": "clientes", "rows": clientes.count()})

In [ ]:
# PRODUTOS
produtos = (
    spark.table(f"{BRONZE_SCHEMA}.produtos")
        .select(
            col("id").cast("string").alias("id"),
            trim(initcap(col("nome"))).alias("nome"),
            trim(initcap(col("categoria"))).alias("categoria"),
            col("preco").cast("double").alias("preco"),
            col("estoque").cast("int").alias("estoque")
        )
        .dropDuplicates(["id"])
        .filter("id IS NOT NULL")
        .filter("nome IS NOT NULL")
        .filter("preco > 0")
        .withColumn(
            "estoque",
            when(col("estoque") < 0, 0).otherwise(col("estoque"))
        )
        .withColumn("_silver_processed_at", current_timestamp())
)

(
    produtos.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{SILVER_SCHEMA}.produtos")
)

results_silver.append({"table": "produtos", "rows": produtos.count()})

In [ ]:
# PEDIDOS
pedidos = (
    spark.table(f"{BRONZE_SCHEMA}.pedidos")
        .select(
            col("id").cast("string").alias("id"),
            col("cliente_id").cast("string").alias("cliente_id"),
            col("produto_id").cast("string").alias("produto_id"),
            col("quantidade").cast("int").alias("quantidade"),
            col("valor_total").cast("double").alias("valor_total"),
            col("data_pedido").cast("date").alias("data_pedido"),
            lower(trim(col("status"))).alias("status")
        )
        .dropDuplicates(["id"])
        .filter("id IS NOT NULL")
        .filter("cliente_id IS NOT NULL")
        .filter("produto_id IS NOT NULL")
        .filter("quantidade > 0")
        .filter("valor_total > 0")
        .withColumn("_silver_processed_at", current_timestamp())
)

(
    pedidos.write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(f"{SILVER_SCHEMA}.pedidos")
)

results_silver.append({"table": "pedidos", "rows": pedidos.count()})

In [ ]:
print("Camada Silver criada com sucesso.")

In [ ]:
for item in results_silver:
    print(f"{item['table']}: {item['rows']} registros")